In [ ]:
# =============================================================================
#  NUCLEAR REACTION EMULATORS -- what you can call, and how
# =============================================================================
#
#   1  pipeline.systems           the benchmarks (physics only)
#   2  pipeline.settings          what a system is studied WITH
#   3  pipeline.calibrate         and what the physics derives for itself
#   4  pipeline.run               declare, plan and execute a computation
#   5  pipeline.compute           the 9 blocks, and their own parameters
#   6  pipeline.cache             results/*.npz, read and write
#   7  pipeline.api               state
#   8  pipeline.api               drawing
#   9  pipeline.api               the numbers behind a figure, and the guardrail
#  10  pipeline.views             arrays -> series
#  11  pipeline.recipes/draw      how a figure looks
#  12  pipeline.observables       solve, sample, convert
#  13  pipeline.metrics           the error definitions
#  14  core.*                     the solver and the emulators
#
#  Every cell below runs. Calls that cost minutes or write to disk are
#  commented out and marked.
#
#  PATHS ARE RELATIVE TO THE WORKING DIRECTORY. cache.RESULTS is Path("results")
#  and figures land in Path("figures_out"), so start Jupyter in this directory
#  or the cache you read is not the cache you think you are reading.
# =============================================================================

# notebook lives in notebooks/ -- put the package root (core/, pipeline/) on the path
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()))

import os

# One BLAS thread: the solves are small and numpy's threading loses badly on
# them -- 38.8 s against 1.0 s on one validation loop. Must precede numpy.
# `import pipeline` does this too, for scripts that never touch this cell.
for _v in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

%load_ext autoreload
%autoreload 2
%matplotlib inline

import dataclasses
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from pipeline import api, cache, calibrate, compute, draw, metrics, observables
from pipeline import recipes, run, settings, systems, views
from core.specfun import SF

print("backends:", ", ".join(SF.names), "    (no 'flint' -> charged systems 11x slower)")
print("cache   :", cache.current().resolve())


In [ ]:
# ============================================================================
#  1. pipeline.systems -- the benchmarks. Physics and mesh, nothing else.
# ============================================================================
#
# SYSTEMS : dict[str, System]   11 entries, the only registry
#
# System fields
#   key           str        its name in SYSTEMS and on disk
#   label         str        LaTeX, for figures
#   preset_fn     callable   () -> (channels, model, theta_c, bounds)
#   N             int        Lagrange mesh points
#   R             float      channel radius, fm
#   E_demo        float      the energy single-energy studies use, MeV
#   E_range       tuple      (E_min, E_max, n_points)
#   coupled       bool       whether the channels talk to each other
#   theta_labels  tuple      LaTeX name of each parameter
#
# System properties (computed, no arguments)
#   E_grid            ndarray  linspace(*E_range)
#   channels          list     the channel dicts
#   n_channels        int
#   n_params          int      len(theta_c)
#   thresholds        list     energies where a channel opens
#   threshold_labels  list
#   absorptive        bool     True if the potential has an imaginary part
#   fingerprint       dict     preset/N/R/E_demo/E_range/channels, for provenance
#
# preset_fn() returns
#   channels  list of dicts   l, j, threshold, mass...
#   model     callable        theta -> (V_diag, V_coup); V_coup is None if uncoupled
#   theta_c   ndarray (p,)    the central parameter vector
#   bounds    ndarray (p, 2)  [lo, hi] per parameter

from pipeline.systems import SYSTEMS

s = SYSTEMS["p12C"]
channels, model, theta_c, bounds = s.preset_fn()
print(list(SYSTEMS)) 

print(f"{s.key}: {s.n_channels} channels x N={s.N} = dim {s.n_channels * s.N}, "
f"R={s.R} fm, {s.n_params} parameters, absorptive={s.absorptive}")

print("theta_c  =", np.round(theta_c, 3))

print("bounds[0] =", bounds[0], "  labels:", s.theta_labels)

print(s.key, s.preset_fn.__doc__)

In [ ]:
# ============================================================================
#  2. pipeline.settings -- the experiment. What a system is studied WITH.
# ============================================================================
#
# Settings(...)                                    frozen dataclass, SIX fields
#   sweep_idx      int      = 0          which parameter F1 sweeps
#   sweep_label    str      = "$V_0$"    its LaTeX name
#   failure_idx    int      = -1         the coupling-only parameter F7 sweeps;
#                                        -1 means no such figure
#   failure_label  str      = ""
#   N_scan         tuple    = (40,60,80,100,120)
#                                        mesh sizes for `invariants` and `scaling`
#   xsec_kind      str|None = None       "elastic" | "reaction" | None =
#                                        whatever the potential allows
#
# NOT settings any more, and the split is the point: ranks, predictor budgets
# and training sizes. They lived here as hand-written tuples -- n_train_rbm,
# n_train_lrom, rbm_nb_list, lrom_configs, K_diag, K_coup -- and they are
# DERIVED now, per system, from the two tolerances of core.calibration. Cell 3.
#
# Nor is the POD tolerance: compute.py fixes eps_tol=1e-14 at every fit site.
#
# declared(key) -> Settings
#     What that system's cache was built with. Raises for a system nothing was
#     ever run on, rather than returning dataclass defaults.
# has_declared(key) -> bool
# xsec_kind(system, settings=None) -> "elastic" | "reaction"
#     settings.xsec_kind if set, else "reaction" when the potential is
#     absorptive and "elastic" when it is real.
#
# SUITE          = ('n40Ca', 'alpha12C_band', 'alpha24Mg', 'n238U')
# CONTROL        = ('p12C', 'alpha_d')     the two control benchmarks
# DECLARED       = dict[str, Settings]     the 6 systems anything has been run on
# FAILURE_SYSTEM = 'alpha24Mg'             F7 is drawn for this one only
#
# DECLARED is a RECORD, not a control: nothing reads it at compute time.
# To run something else, pass your Settings to a Run (cell 4).

from pipeline.settings import (CONTROL, DECLARED, FAILURE_SYSTEM, SUITE,
                               Settings, declared, has_declared, xsec_kind)

st = declared("n40Ca")
mine = dataclasses.replace(st, sweep_idx=3, sweep_label=r"$V_{\rm so}$")
print("declared :", st)
print("mine     :", mine)
print("suite    :", SUITE, " control:", CONTROL, " declared for:", sorted(DECLARED))
print("reports  :", xsec_kind(SYSTEMS["n40Ca"]),
      "| forced:", xsec_kind(SYSTEMS["n40Ca"], Settings(xsec_kind="elastic")))
print("has_declared('o16Ca44'):", has_declared("o16Ca44"))


In [ ]:
# ============================================================================
#  3. pipeline.calibrate -- the hyperparameters, read off the physics
# ============================================================================
#
# calibrate.probe(system_key, eps_pot=EPS_POT, eps_sol=EPS_SOL, n_probe=120,
#                 seed=0, verbose=False) -> dict
#     Solves n_probe draws, takes the SVD of the snapshots and of the potential
#     terms, and reads the ranks off the spectra. eps_pot is the tolerance on
#     the potential (it sets K_diag/K_coup), eps_sol the one on the solution
#     (it sets the rank nb). 10-90 s depending on the system: a real calculation.
#
# plan_calibration.calibration_for(key, **probe_kw) -> Calibration
#     The probe as a VALUE, memoised per process AND FROZEN TO DISK in
#     results/_calibration.json. The second half is what makes the experiment
#     reproducible: nb, K and N_s configure every block, they were re-derived by
#     SVD at every session, and nothing recorded or compared them -- so a numpy
#     upgrade could move the rank without a single block being marked stale.
#     The stored entry is keyed on the two tolerances, the probe size and seed,
#     and the system's own fingerprint, so a mesh that moves re-probes and
#     nothing else does.
#
# plan_calibration.calibrations() -> DataFrame
#     What is frozen, and under which numpy and which special-function chain.
# plan_calibration.recalibrate(key, **probe_kw) -> Calibration
#     Re-derive one system and overwrite the stored entry. Deliberate, like
#     api.freeze(): a calibration that moves moves every figure of that system.
#
# Calibration
#   .system .nb .K_diag .K_coup .Ns .eps_sol .eps_pot .n_probe
#   .K                 K_diag + K_coup
#   .r_fit             Ns / [K (nb+1)], Eq. (24). Below 1 the LROM's
#                      least-squares system is wider than tall and lstsq
#                      returns the minimum-norm interpolant.
#   .well_posed_rank   the largest rank this budget supports with r_fit >= 1
#   .ladder(n=6)       n log-spaced ranks from nb/4 to nb, top rung = nb
#   .budgets(n=2)      the top n (K_diag, K_coup) pairs
#
# One ladder, and each block takes what its figure needs off the top of it:
# `manifold` the whole of it, `cat` two rungs and two budgets, `excitation` and
# `interior` one of each. as_meta(cal) is stored beside every block's arrays,
# so a cached result can always be asked what it was made under.
#
# What a run will COST is Run(...).plan() in cell 4, which also reads the cache.

from pipeline.plan_calibration import (as_meta, calibration_for, calibrations,
                                       recalibrate)

cal = calibration_for("n40Ca")          # probed once, then read from disk
print(cal)
display(calibrations())
print("  ladder :", cal.ladder())
print("  budgets:", cal.budgets(2))
print("  meta   :", as_meta(cal))


In [ ]:
# ============================================================================
#  4. pipeline.run -- the only way to compute anything
# ============================================================================
#
# Run(systems=SUITE, settings=None, methods=('rbm','lrom'),
#     blocks=None, force="if-changed")
#   systems   list[str]          keys of SYSTEMS
#   settings  None               = declared(key), what the cache was built with
#             Settings           = that one, for every system
#             dict[str, ...]     = per system; a missing key falls back to
#                                  declared()
#             a STRING raises. Ranks and budgets are no longer a run-level
#             choice -- every block reads pipeline.plan_calibration.
#   methods   list[str]          subset of METHODS = ('rbm', 'lrom'); anything
#                                else raises. Only METHOD_BLOCKS =
#                                ('excitation', 'interior') read it.
#   blocks    list[str] or None  None = all nine, in cache.BLOCKS order
#   force     "never"            compute only what is absent
#             "if-changed"       and what was made under other settings, or on
#                                another mesh
#             "always"           everything
#
# .plan() -> DataFrame           what WOULD happen. Nothing runs.
#     cached   what is on disk       action  compute|recompute|extend|skip
#     methods  what will be fitted   why     the reason, field by field
#     est_s    seconds, order of magnitude, floored at 5
# .execute(progress=True) -> DataFrame   runs it; progress=False for no bar
# .settings_for(key) -> Settings         resolved once per run
#
# "if-changed" is a comparison, not a guess: every block stores the settings AND
# the system fingerprint (mesh, radius, channel count) it was produced under, so
# the plan names the field that moved instead of shrugging and recomputing.
#
# inventory(systems=None) -> DataFrame
#     What the cache holds, read from the files. `methods` comes from the
#     ARRAYS, not the metadata: a full-size sigma_et of nothing but NaN counts
#     as absent, and this is where you see that.

from pipeline.run import METHOD_BLOCKS, METHODS, Run, inventory

display(inventory(["n40Ca", "alpha24Mg"]))

Run(systems=["n40Ca"], blocks=["manifold", "cat"], methods=["rbm", "lrom"],
    force="if-changed").plan()

# .execute() is what writes:
# Run(systems=["p12C"], blocks=["sweep", "validation"], force="never").execute()


In [ ]:
# ============================================================================
#  5. pipeline.compute -- the nine blocks
# ============================================================================
#
# Every one takes (system, settings) -- a System object and a Settings -- and
# writes one results/<system>/<block>.npz. Run calls them; call them directly
# only to override the extra parameters, which no Settings field controls.
#
# Ranks, predictor budgets and training sizes are in NEITHER argument: each
# block reads plan_calibration.calibration_for(system.key) for itself.
#
# block_sweep(system, settings, n_sweep=9, n_box=200)
#     -> F0, F1. The interaction term by term, one parameter walked across its
#     range (n_sweep curves), and the p5-p95 envelope over n_box Latin-hypercube
#     draws of the full box.
#
# block_validation(system, settings, n_theta=40, n_E=60)
#     -> F2, T1. DBMM against the R-matrix reference on sigma, on S and on
#     chi(r), at n_theta draws x n_E energies.
#
# block_invariants(system, settings)
#     -> F3. Symmetry and unitarity across settings.N_scan, with and without the
#     flux factor, plus the ratio that names the missing factor.
#
# block_manifold(system, settings, n_valid=40)
#     -> F4, F5. POD spectrum, error by rank over cal.ladder(), error by
#     quantity, error against sample size.
#
# block_failure(system, settings, n_train=400, n_sweep=25, K_total=12, nb=32)
#     -> F7. One LROM budget K_total given entirely to the diagonal, then split
#     in half, along settings.failure_idx. Only run for FAILURE_SYSTEM.
#
# block_excitation(system, settings, methods=('rbm','lrom','rbm_et'), n_E=20,
#                  n_resid=30, n_rmat=12)
#     -> F8, F5b. sigma(E) end to end, one offline stage per energy node.
#     n_resid draws for the error band, n_rmat energies checked against the
#     R-matrix code. MERGES: methods not asked for survive on disk.
#
# block_interior(system, settings, methods=('rbm','lrom'), n_E=20, n_resid=30)
#     -> F9, F10. psi(r) at E_demo and delta(E) over the grid, solver against
#     the emulators. Merges, like excitation.
#
# block_cat(system, settings, n_valid=150, n_rbm=2, n_lrom=3)
#     -> F6b, F6, T2. The cost-accuracy cloud: one row per (configuration,
#     validation draw), every draw keeping its own measured time and error.
#
# block_scaling(system, settings, n_valid=30)
#     -> F11. Online cost of both emulators against mesh size, physics fixed.
#
# N_ENERGY_NODES = 20   the n_E default of the two blocks that refit per energy,
#                       and the one number that sets the price of a run
# BLOCK_FUNCTIONS : dict[str, callable]   name -> the function above

print(list(compute.BLOCK_FUNCTIONS))
print("N_ENERGY_NODES =", compute.N_ENERGY_NODES)

# WRITES results/n40Ca/invariants.npz, a few seconds at this size:
# compute.block_invariants(SYSTEMS["n40Ca"], declared("n40Ca"))


In [ ]:
# ============================================================================
#  6. pipeline.cache -- results/<system>/<block>.npz
# ============================================================================
#
# BLOCKS : dict[str, str]   the nine names -> which figures they feed
#
# path(system_key, block) -> Path
#
# read(system_key, block) -> (arrays: dict[str, ndarray], meta: dict)
#     Raises FileNotFoundError naming the Run that would produce it.
#     meta always carries: system, block, written, core_fingerprint, numpy,
#     python, blas, provenance {settings, system}; and for excitation and
#     interior a methods {name: {settings, system, seconds}} record per emulator.
#
# write(system_key, block, arrays, meta=None, methods=None, merge=False) -> Path
#     merge=True reads the block back first and adds to it, which is what lets
#     LROM be computed later without destroying the RBM results.
#
# require(system_keys, block) -> dict[str, (arrays, meta)]
#     Reports every missing system at once instead of failing on the first.
#
# methods_present(block, arrays, meta=None) -> list[str]
#     Which emulators the block really holds, read from the arrays -- an
#     all-NaN sigma_et counts as absent. Empty for the seven blocks with no
#     method axis.
#
# use(name=None) / current() / name() / available()
#     Switch every read and write to results-<name>/, and the figures to
#     figures_out/<name>/, so a second configuration can be computed and drawn
#     without destroying the one the report is built on. api.use_cache() wraps
#     this and shows what is in each.

print("blocks:", list(cache.BLOCKS))
arrays, meta = cache.read("n40Ca", "manifold")
print("arrays :", list(arrays))
print("meta   :", {k: v for k, v in meta.items() if k != "provenance"})
print("methods:", cache.methods_present("excitation",
                                        *cache.read("alpha24Mg", "excitation")))


In [ ]:
# ============================================================================
#  7. pipeline.api -- state
# ============================================================================
#
# api.systems(keys=None) -> DataFrame
#     One row per benchmark: channels, N, dim, R, E_demo, E_range, coupled,
#     absorptive, reports, params, thresholds, computed.
#     keys = None -> all eleven.
#
# api.settings(keys=None) -> DataFrame
#     One row per system anything has been run on. The declared half is `swept`
#     and `reports`; the derived half -- nb, K_diag, K_coup, Ns, r_fit, ranks --
#     is probed from that system's own spectra and written by nobody.
#     keys = None -> the six in DECLARED. Probes, so the first call is slow.
#
# api.figures() -> DataFrame
#     One row per recipe: subject, scope (suite | overlay | per-system),
#     panels, drawn / of, and the output filename.
#
# api.caches() / api.use_cache(name=None) -> DataFrame
#     Which cache directories exist, how full they are, and which is in force.

display(api.systems(["n40Ca", "alpha12C_band"]))
display(api.figures())
api.settings(["n40Ca", "alpha24Mg"])          # probes: ~30 s cold, then cached


In [ ]:
# ============================================================================
#  8. pipeline.api -- drawing
# ============================================================================
#
# In every call below `recipe` is a key of api.figures(), e.g.
# "vs_rmatrix.suite" -- NOT a view name. Views are cell 10.
#
# api.figure(recipe, systems=None, methods=None) -> Path | list[Path]
#     Draws and returns where it was written. A per-system recipe returns one
#     path per system.
#     systems = None -> the recipe's own `systems` if it declares any, else SUITE
#     methods = None -> whatever the recipe declares
#
# api.show(recipe, systems=None, methods=None, report=True) -> same
#     figure(), displayed inline. report=True prints what could not be drawn.
#
# api.can_draw(recipe, systems=None, methods=None) -> DataFrame
#     One row per (system, method): state = drawn | empty | absent, and why.
#     Ask BEFORE drawing: a method requested and missing gives an empty panel,
#     which reads like a converged emulator rather than like absent data.
#
# api.all_figures(systems=None) -> dict[str, Path | str]
#     Every recipe. A value that is a string is the error text.

display(api.can_draw("observable_vs_emulators.suite",
                     systems=["n40Ca", "alpha24Mg"],
                     methods=["rbm", "lrom", "rbm_et"]))

api.show("observable_vs_emulators.suite", systems=["n40Ca", "alpha24Mg"],
         methods=["rbm", "lrom"])

# for recipe, out in api.all_figures().items():   # ~20 s
#     print(f"{recipe:38s} {out}")

In [ ]:
# ============================================================================
#  9. pipeline.api -- the numbers behind a figure, and the guardrail
# ============================================================================
#
# api.view(name, system) -> dict[str, PanelData]
#     `name` is a VIEW name, without the ".suite" suffix. Cell 10 lists them.
#
# api.notes(name, system) -> DataFrame
#     Every panel's summary values, as (panel, note, value) rows.
#
# api.check(path="tests/reference_blocks.json") -> str | DataFrame
#     Whether any cached NUMBER moved since the reference was frozen. Not the
#     same question as "did the code change". Timings are excluded: they move a
#     few per cent every run. New blocks are reported as `new`, not as changes.
#
# api.freeze(path="tests/reference_blocks.json") -> str
#     Accept the current numbers as the reference. Deliberate: freezing to make
#     a check pass is how a regression becomes the baseline.

display(api.notes("vs_rmatrix", "alpha24Mg"))
print(api.check())

# api.freeze()

In [ ]:
# ============================================================================
#  9b. api.tables / api.manifest / api.report -- from a cache to the memoir
# ============================================================================
#
# api.tables(systems=None) -> DataFrame
#     Writes tables_out/*.tex from the cache: t_manifold, t_cost, t_interior and
#     t_macros. Every one is built from the same `views` call the neighbouring
#     FIGURE is drawn from, so a table and the figure beside it cannot disagree.
#     They used to be typed by hand into 05b_results_body.tex and had already
#     drifted -- the manifold table quoted a rank selected at a tolerance the
#     calibration does not use.
#
#     In the document, once per table:
#         \input{../nuclear_emulators/tables_out/t_manifold.tex}
#     and for the numbers quoted in prose:
#         \input{../nuclear_emulators/tables_out/t_macros.tex}
#         ... falls at $N^{\ast}=\NstarRbmNFZCa$ evaluations ...
#     (no digits in TeX control sequences, so n40Ca becomes NFZCa.)
#
# api.stale() -> str | dict
#     What is older than what it was built from, at both levels of the chain:
#
#         core/ + compute.py  ->  blocks  ->  views/draw/recipes  ->  png, tex
#
#     `recompute` compares each block's recorded core_fingerprint against the
#     current sources -- a CONTENT hash, so it is exact. `redraw` compares
#     modification times, so it is a warning: a docstring edit moves views.py
#     without moving a pixel. When it fires, redraw and compare; rendering is
#     deterministic and a full pass costs about a minute.
#
#     f7_failure.png is expected to be listed. Its block is deliberately not
#     computed, so nothing can refresh it.
#
# api.rebench(systems=None) -> dict
#     Re-measure what depends on the MACHINE, and only that. A new interpreter,
#     another laptop, a BLAS rebuild: none of them can move an accuracy number
#     and all of them move every timing. Recomputing everything for that is 52
#     minutes to change 26 per cent of it, and it puts the physics through a
#     rerun it did not need.
#
#     cache.MACHINE_DEPENDENT = ('cat', 'scaling') is where every timing in the
#     cache lives -- tests/test_outputs.py asserts no other block stores one --
#     so this recomputes those two, redraws exactly the five recipes whose view
#     reads them, and regenerates t_cost and t_macros. 13 min against 52.
#
#     draw.VIEW_BLOCKS is the map it uses: which block each view reads, declared
#     as data and checked against what the views really open.
#
# api.manifest(path=None) -> Path
#     One results/_manifest.json describing the whole state: interpreter, numpy,
#     BLAS threads, the special-function chain, the Coulomb cache, every frozen
#     calibration, the hash of every cached array, and which figures and tables
#     are on disk. Per-block provenance answers "what made this array"; this
#     answers "what made this PDF".
#
# api.report(systems=None, force="if-changed", compute=True, methods=None)
#     compute -> draw -> tabulate -> record -> check, in that order and in one
#     call. Returns a dict of tables to look at. compute=False redraws from the
#     cache as it stands, which is what you want while iterating on a figure.
#
# THE ORDER MATTERS. Tables come from the same views as the figures, the
# manifest is written after both so it describes what is really on disk, and the
# guardrail runs last so a number that moved is the last thing printed.

display(api.tables())
print(api.manifest())
print(api.stale())

# After a machine or interpreter change, this and nothing else:
# out = api.rebench()

# The whole thing, computing what is missing (hours on the full suite):
# out = api.report()
# for name, value in out.items():
#     print(f"--- {name} ---"); display(value)


In [ ]:
# ============================================================================
#  10. pipeline.views -- cached arrays -> plot-ready series. No matplotlib.
# ============================================================================
#
# Every view takes one argument, system_key, and returns dict[str, PanelData].
# The panel names are the `rows` a recipe may ask for:
#
#   validation                ('main', 'residual')        <- api.view: vs_rmatrix
#   potential                 ('potential',)
#   manifold_spread           ('eigenphase', 'wavefunction', 'xsec')
#   pod_spectrum              ('spectrum', 'error_vs_ns')
#   error_by_rank             ('error',)
#   error_by_quantity         ('errors', 'magnitude')
#   error_distribution        ('distribution',)
#   observable_vs_emulators   ('sigma', 'error')
#   wavefunction_vs_emulators ('psi', 'residual')
#   phase_shift_vs_emulators  ('delta', 'residual')
#   cat_cloud                 ('cloud',)
#   cat_cloud_inelastic       ('cloud',)      the same view on |S_ab|
#   amortisation              ('breakeven',)
#   online_vs_mesh            ('online', 'ratio')
#   invariants                ('symmetry', 'unitarity', 'ratio')
#   coupling_budget_failure   ('elastic', 'inelastic', 'errors', 'predictors')
#
# draw.VIEWS maps the name api.view() takes to the function. F7 returns an
# 'elastic' panel its recipe does not draw: it plotted delta over a 0.1 degree
# window and argued against the figure's own claim, so it is kept readable
# through api.view and left off the sheet.
#
# PanelData(key, title, series=[], x_label="", y_label="", thresholds=[],
#           threshold_labels=[], notes={}, provenance={})
#   notes       the panel's own summary numbers; recipes pull headlines,
#               hlines, vlines and axis floors out of here by name
#   provenance  what the block was computed under; the renderer turns the four
#               columns' worth into the one line under the sheet
# Series(x, y, role, label=None, band=None, style="", point_labels=None)
#   role   how it is drawn: reference | comparison | residual | cloud | scatter
#          | front | dominated | shape | ticks | span | sweep
#   band   (lo, hi) filled around the curve, or (value,) for a single marker
#   style  which pen: a method (rbm, lrom, rbm_et), a quantity (elastic,
#          inelastic, sigma, norm), or a weight (thin, thick, achieved, ...)

print(list(draw.VIEWS))
panels = api.view("vs_rmatrix", "alpha24Mg")
print(list(panels))
for series in panels["main"].series:
    print(f"  {series.label:10s} role={series.role:11s} style={series.style:9s} "
          f"{len(series.x):3d} pts  y in [{series.y.min():.3g}, {series.y.max():.3g}]")
display(api.notes("online_vs_mesh", "n40Ca"))


In [ ]:
# ============================================================================
#  11. pipeline.recipes / pipeline.draw -- how a figure looks
# ============================================================================
#
# RECIPES : dict[str, Recipe]   18 figures, plus a generated control twin for
#                               12 of them = 30 entries
#
# Recipe fields
#   key                 str    "<view>.<variant>"; the part before the dot
#                              picks the view
#   stem                str    output filename, without .png
#   rows                tuple  which panels of the view, top to bottom
#   height_ratios       tuple  per row; () = equal
#   figsize             tuple  inches
#   y_scale / x_scale   tuple  "log" | "linear" | "sqrtlog", per row
#   share_y             bool   one y axis per row across systems
#   overlay_systems     bool   all systems in one panel, colour = system
#   per_system          bool   one sheet per system, figures_out/<sys>_<stem>.png
#   systems             tuple  which systems when the caller does not say;
#                              () = SUITE
#   methods             tuple  which methods survive into the sheet; () = all
#   mark_thresholds     bool   vertical lines where a channel opens
#   tolerance_ladder    tuple  rows carrying the 5% / 0.5% lines
#   headlines           dict   row -> (note, format[, corner]), inside the panel
#   hlines / vlines     dict   row -> (note, label)
#   end_labels          dict   row -> (note, format), at the end of the curve
#   orientation         dict   row -> which way is better
#   weight_legend       dict   style key -> what that line weight means
#   legend_in           int    which column carries the legend
#   legend_all_columns  bool   every column names different curves (F0 does)
#   legend_loc          dict   row -> matplotlib loc, when "best" gets it wrong
#   caption / suptitle  str
#
# Every one of those is a LITERAL: changing a figure is an edit to a value, and
# nothing is recomputed. A redraw costs ~2 s.
#
# draw.render(recipe, system_keys, methods=None) -> Path | list[Path]
#     Takes a Recipe OBJECT, not a name. This is what api.figure calls.
# draw.audit(recipe, system_keys, methods=None) -> list[dict]
#     What api.can_draw tabulates.
# draw.out_dir() -> Path
#     figures_out/, or figures_out/<name>/ when a named cache is in force.

from pipeline.recipes import RECIPES

print(f"{len(RECIPES)} recipes")

variant = dataclasses.replace(
    RECIPES["vs_rmatrix.suite"],
    stem="scratch",                 # -> figures_out/scratch.png, original untouched
    height_ratios=(2.0, 2.0),
    mark_thresholds=False,
)
display(Image(filename=str(draw.render(variant, ["n40Ca", "alpha24Mg"]))))


In [ ]:
# ============================================================================
#  12. pipeline.observables -- solve, sample, convert
# ============================================================================
#
# solver_for(system, theta, energies=None, N=None, R=None) -> DBMMSolver
#     Builds the solver at parameter vector theta. Passing `energies` warms the
#     Coulomb cache for exactly those energies, which is most of the cost on a
#     charged system. N and R override the system's mesh.
#
# sample_theta(system, n, seed, include_center=True) -> ndarray (n, p)
#     Latin hypercube over the system's bounds. include_center=True puts
#     theta_c in row 0 -- pass False for a validation set, or the emulator is
#     scored at a point it was trained on.
#
# cross_section(solver, E, S, kind) -> float
#     kind = "elastic" | "reaction". Takes the S it is given, so one solve can
#     be converted several ways.
#
# eigenphase_branch(S_grid, entrance=0) -> (delta_deg, modulus)
#     Follows one eigenvalue of S along the grid and unwraps it. It is a
#     property of the GRID, not of a point: the same S-matrix reads 26.95 deg
#     inside a varying grid and 36.03 deg presented as a constant one. Only use
#     it along an axis the solution varies smoothly in, and never to compare
#     two grids computed separately.
#
# open_channels(system, E) -> list[int]     channels above threshold at E
# energy_windows(system) -> list[(lo, hi)]  the intervals between thresholds
# is_physical(S, tol=0.1) -> bool           largest singular value <= 1 + tol

from pipeline.observables import (cross_section, eigenphase_branch,
                                  energy_windows, is_physical, open_channels,
                                  sample_theta, solver_for)

sysm = SYSTEMS["alpha24Mg"]
E = np.linspace(10.0, 80.0, 60)                 # E = 0 gives k = 0, so stay above it
solver = solver_for(sysm, sysm.preset_fn()[2], energies=E)
S = [solver.solve(e) for e in E]

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True)
for kind in ("elastic", "reaction"):
    ax[0].plot(E, [cross_section(solver, e, Se, kind) for e, Se in zip(E, S)],
               label=kind)
ax[0].set_yscale("log"); ax[0].set_ylim(bottom=1e-6); ax[0].legend(frameon=False)
ax[0].set_xlabel(r"$E$ (MeV)"); ax[0].set_ylabel(r"$\sigma$ (fm$^2$)")
ax[1].plot(E, eigenphase_branch(np.array(S))[0])
ax[1].set_xlabel(r"$E$ (MeV)"); ax[1].set_ylabel(r"$\delta$ (deg)")
plt.show()

print("windows        :", energy_windows(sysm))
print("open at 54 MeV :", open_channels(sysm, 54.0), "of", sysm.n_channels)
print("physical       :", is_physical(S[-1]))

In [ ]:
# ============================================================================
#  13. pipeline.metrics -- the error definitions
# ============================================================================
#
# S arrays are (n_draws, Nc, Nc); every metric reduces the trailing two axes
# and returns ONE VALUE PER DRAW, not a scalar. Take a median yourself -- the
# project's convention, because the max is set by whichever matrix element sits
# nearest zero.
#
# err_global(S_emu, S_ref)                  ||dS||_F / ||S||_F
# err_elastic(S_emu, S_ref)                 worst relative error on the diagonal
# err_inelastic(S_emu, S_ref, mask=None)    ... on the off-diagonal elements that
#                                           exist; NaN when none do
# err_report(S_emu, S_ref, mask=None)       what the report scores: inelastic
#                                           where coupling exists, elastic otherwise
# err_observable(x_emu, x_ref)              relative error on a real quantity
# coupling_mask(S_ref, rel_floor=1e-08)     (Nc, Nc) bool: which off-diagonal
#                                           elements are genuinely non-zero
#
# symmetry_violation(S)                     max |S_ab - S_ba|
# unitarity_violation(S, open_idx=None, closed_floor=1e-12)
#                                           ||S^H S - I||_F on the OPEN sub-block.
#                                           Large on an absorptive potential: flux
#                                           IS lost. Only ~0 for a real one.
# wavefunction_residual(psi_a, psi_b)       max |dpsi| / max |psi_a|, per draw
# residual_stats(x)                         median / p95 / max, spread included
#
# break_even(t_offline_a, t_online_a, t_offline_b=0.0, t_online_b=None) -> float
#     How many evaluations before A overtakes B in T_off + N * T_on. inf when A
#     is never cheaper. t_online_b is required despite the default.
# pareto_front(cost, error)                 indices of the non-dominated points
# operating_point(errors, costs, tol=0.005) cheapest configuration under tol
# is_efficient(errors, costs, t_offline, t_online_ref, n_campaign, tol=0.005)

th = sample_theta(sysm, 4, seed=7, include_center=False)
S_ref = np.array([solver_for(sysm, t, energies=[54.0]).solve(54.0) for t in th])
S_emu = S_ref * (1 + 1e-3)                       # a fake emulator, 0.1% off

mask = metrics.coupling_mask(S_ref)
for name, value in (
        ("err_global", metrics.err_global(S_emu, S_ref)),
        ("err_elastic", metrics.err_elastic(S_emu, S_ref)),
        ("err_inelastic", metrics.err_inelastic(S_emu, S_ref, mask=mask)),
        ("err_report", metrics.err_report(S_emu, S_ref, mask=mask)),
        ("symmetry_violation", metrics.symmetry_violation(S_ref)),
        ("unitarity_violation", metrics.unitarity_violation(S_ref))):
    value = np.atleast_1d(value)
    print(f"{name:20s} median {np.nanmedian(value):.2e}   {value.size} draws")

print(f"{'break_even':20s} {metrics.break_even(12.0, 0.0002, 0.0, 0.007):.0f} "
      f"evaluations before a 12 s fit pays for itself")

In [ ]:
# ============================================================================
#  14. core -- the solver and the four emulators
# ============================================================================
#
# DBMMSolver(N, R, channels, V_diag, V_coup=None)
#   .solve(E) -> ndarray (Nc, Nc)     the collision matrix
#   Use observables.solver_for unless you are overriding the potential itself.
# flux_factor_disabled()              context manager; drops the velocity factor,
#                                     which is what F3 measures
#
# latin_hypercube(n_samples, bounds, seed=0) -> ndarray (n_samples, p)
#
# All four emulators: build, fit once (offline), predict many times (online).
# `model` is preset_fn()[1]; `base_solver` is a solver built at theta_c.
# eps_tol is the POD tolerance, nb_max caps the rank -- the binding one wins,
# and .nb afterwards says what was actually kept.
#
# RBMEmulator(base_solver, E)                                  one energy
#   .fit(model, theta_train, eps_tol=1e-08, nb_max=None)
#   .predict(theta) -> S
#
# LROMEmulator(base_solver, E)                                 one energy
#   .fit(model, theta_train, theta_c, K_diag=6, K_coup=6, eps_tol=1e-08,
#        nb_max=None, predictor_points=None)
#   .predict(theta) -> S
#   .points                                        (kind, a, b, r, channels) per point
#   K_diag/K_coup are predictor points on the diagonal / coupling blocks.
#   Needs n_train >= (K_diag + K_coup) * (nb + 1), or the residual fit is
#   underdetermined -- it warns, and the run below trips that on purpose at
#   nb = 16 and 32.
#   Spend nothing on the coupling and the emulator cannot see a coupling-only
#   parameter at all: its prediction is then CONSTANT in it. That is F7.
#
# RBMEmulatorET(base_solver)                        transferable in energy
#   .fit(model, theta_train, E_train, eps_tol=1e-08, nb_max=None)
#   .predict(theta, E) -> S
#   E_train must stay inside ONE energy window (observables.energy_windows):
#   across a threshold the number of open channels changes and a single basis
#   cannot span both sides.
#
# RBMEmulatorETP(base_solver)                       ... and parametric
#   .fit(model, theta_train, E_train, theta_c, eps_tol=1e-08, nb_max=None,
#        K_diag=6, K_coup=0)
#   .predict(theta, E) -> S

from core.rbm import RBMEmulator, latin_hypercube
from core.lrom_dbmm import LROMEmulator

sysm = SYSTEMS["n40Ca"]
channels, model, theta_c, bounds = sysm.preset_fn()
E_demo, kind = float(sysm.E_demo), xsec_kind(sysm)

th_train = sample_theta(sysm, 120, seed=1)
th_valid = sample_theta(sysm, 12, seed=999, include_center=False)
base = solver_for(sysm, theta_c, energies=[E_demo])
S_ref = np.array([solver_for(sysm, t, energies=[E_demo]).solve(E_demo)
                  for t in th_valid])
sig_ref = np.array([cross_section(base, E_demo, S, kind) for S in S_ref])

rows = []
for nb_max in (4, 8, 16, 32):
    for name in ("rbm", "lrom"):
        t0 = time.perf_counter()
        if name == "rbm":
            emu = RBMEmulator(base, E_demo)
            emu.fit(model, th_train, eps_tol=1e-14, nb_max=nb_max)
        else:
            emu = LROMEmulator(base, E_demo)
            emu.fit(model, th_train, theta_c, K_diag=8, K_coup=0,
                    eps_tol=1e-14, nb_max=nb_max)
        t_off = time.perf_counter() - t0

        t0 = time.perf_counter()
        S_emu = np.array([emu.predict(t) for t in th_valid])
        t_on = (time.perf_counter() - t0) / len(th_valid)

        sig = np.array([cross_section(base, E_demo, S, kind) for S in S_emu])
        rows.append(dict(method=name, nb_max=nb_max, nb=emu.nb,
                         err=float(np.median(metrics.err_observable(sig, sig_ref))),
                         offline_s=round(t_off, 3), online_ms=round(1e3 * t_on, 3)))

t0 = time.perf_counter()
for _ in range(20):
    base.solve(E_demo)
print(f"solver: {1e3 * (time.perf_counter() - t0) / 20:.3f} ms per point, "
      f"to compare against online_ms below")

pd.DataFrame(rows).set_index(["method", "nb_max"])

In [ ]:
# ============================================================================
#  FILES
# ============================================================================
#
#  core/       constants.py  dbmm.py  python_rmatrix.py  rbm.py  lrom_dbmm.py
#              rbm_et.py  potentials.py  specfun.py  calibration.py
#  pipeline/   systems.py  settings.py  presets.py  run.py  compute.py
#              cache.py  views.py  recipes.py  draw.py  observables.py
#              metrics.py  coulomb.py  style.py  calibrate.py
#              plan_calibration.py  reference.py  tables.py  api.py
#  results/    the cache: <system>/<block>.npz, plus
#              _calibration.json   the frozen hyperparameters, per system
#              _manifest.json      what produced everything (api.manifest)
#              _coulomb_cache.pkl  derived, stamped with the specfun chain
#  figures_out/  the PNGs        tables_out/  the .tex the memoir \input's
#  tests/      python -m pytest tests/ -q
#  NOTES_baselines.md   what the speedups are measured against, and which of
#                       the three baselines is biased. Read it before quoting
#                       any ratio whose denominator is t_rmat.
#  xsec_scan.py, xsec_methods.py    standalone scripts, no notebook
#
#  WHERE THINGS ARE WRITTEN
#   Everything is relative to the working directory: cache.RESULTS is
#   Path("results"), draw.OUT is Path("figures_out"), tables.OUT is
#   Path("tables_out"). Start Jupyter here. cache.use("hires") moves all three,
#   so a second configuration cannot overwrite the one the report is built on.
#
#  KNOWN, AND DELIBERATE
#   * The R-matrix column of the cost table is an UPPER BOUND, not the cost of
#     the method: core/python_rmatrix.py rebuilds its theta-independent kinetic
#     matrix on every call and inverts the full matrix where Nc right-hand sides
#     would do. Measured, and left unfixed on purpose so that no published
#     number moves without the text moving with it. NOTES_baselines.md.
#   * core.specfun.NumbaBackend is written, tested and OUT of DEFAULT_CHAIN. It
#     is correct to 2e-12 where the default chain is correct to 1.9e-16, and the
#     smallest accuracy the report quotes is 4.3e-9.
#   * pod_basis thresholds the CUMULATIVE ENERGY, and 1 - eps_tol saturates at
#     1.0 below eps_tol ~ 1e-16: the rank cannot be pushed past ~43 on n+40Ca
#     through the tolerance. Irrelevant in production, where the calibrated
#     nb_max binds first, and worth knowing before trying.
#
#  NOT AVAILABLE, so you do not go looking
#   * rbm_et cannot be reached through a Run: METHODS is ('rbm','lrom') and the
#     constructor rejects anything else. block_excitation still fits it if you
#     call it yourself with methods=("rbm_et",).
#   * F7 (coupling_budget_failure) needs results/alpha24Mg/failure.npz, which is
#     not in the cache. It is deliberately out of the memoir.
#   * 5 of the 11 systems have never been computed and have no declared
#     settings: n58Ni, p40Ca, n40Ca_coupled, alpha208Pb, o16Ca44. Give them a
#     Settings() of your own and a Run will compute them.
#   * tests/reference_blocks.json still predates the block rewrite: it names
#     `cost`, which is now `cat`, and knows nothing of `scaling` or `interior`.
#     api.check() therefore reports the whole cache as changed. Re-freeze it
#     with api.freeze() AFTER the definitive recompute, never before.

print("see the comments above")
